# Note

This is for UMass COMPSCI 485:

Once done after class, please save as PDF and submit on Gradescope as an exercise.


# Setup

let's get started with Google Colab and the Huggingface libraries!

Google Colab is just a hosted version of Jupyter Notebook.  It offers, sometimes, free GPUs, which are very useful for neural net-based work.

In [1]:
# This is a virtual machine. In the cloud!
!pwd

/content


In [2]:
!ls -laF

total 16
drwxr-xr-x 1 root root 4096 Apr 16 13:28 ./
drwxr-xr-x 1 root root 4096 Apr 21 16:59 ../
drwxr-xr-x 4 root root 4096 Apr 16 13:28 .config/
drwxr-xr-x 1 root root 4096 Apr 16 13:28 sample_data/


## Adding a hardware accelerator
The purpose of this homework is to get you acquainted with using large-scale pretrained language models specifically in the context of transfer learning. When you're training large neural networks, it's good to attach a GPU, otherwise training will take a very long time.

In class we didn't need to do the following.  But if the GPU isn't enabled for Google Colab, go to the menu and add a GPU as follows:

`Edit > Notebook Settings > Hardware accelerator > (GPU)`

Or use the "GPU" button on the top-right.  Run the following cell to confirm that the GPU is detected.

In [3]:
import torch

# Confirm that the GPU is detected

assert torch.cuda.is_available()

# Get the GPU device name.
device_name = torch.cuda.get_device_name()
n_gpu = torch.cuda.device_count()
print(f"Found device: {device_name}, n_gpu: {n_gpu}")

AssertionError: 

## Installing Hugging Face's Transformers library
We will use Hugging Face's Transformers (https://github.com/huggingface/transformers), an open-source library that provides general-purpose architectures for natural language understanding and generation with a collection of various pretrained models made by the NLP community. This library will allow us to easily use pretrained models like `BERT` and perform experiments on top of them. We can use these models to solve downstream target tasks, such as text classification, question answering, and sequence labeling.

In [ ]:
# These should be pre-installed on Google Colab
#!pip install tokenizers
#!pip install transformers

# BERT

We'll use `BERT` [(Devlin et al., 2019)](https://arxiv.org/pdf/1810.04805.pdf) for the task of masked word completion: given an input sentence with some words masked out, predict the masked word(s) based on its context. Run the following cell to download the pretrained `BERT` base model (cased) and tell PyTorch to use the GPU to run it.

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForMaskedLM

modelname = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(modelname)
model = AutoModelForMaskedLM.from_pretrained(modelname)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print('done!')

In [ ]:
tokenizer.vocab

In [ ]:
model

# Tokenization

In [ ]:
sentence = f"""u.s."""

# Encode the input sentence and get the model's output
input = tokenizer.encode(sentence, return_tensors="pt").to(device)
print(input)
[tokenizer.decode([x]) for x in input[0]]

# Activity 1

Find at least five other words that get subtokenized.  
Do they have anything in common?
Mark them down here:

# MLM predictions
Let's get MLM predictions for a masked word.

*Hints*

*   Use `torch.where` to find the index of a masked token within the input tensor (note that `tokenizer.mask_token_id` gives us the index of the mask token in the vocabulary).
*   Use `torch.topk` to get the `k` largest elements of a given tensor along a given dimension.
*   Use `tokenizer.decode([token_id])` to convert a single integer `token_id` to a token string.



In [ ]:
tokenizer.mask_token

In [ ]:
sentence = f"""My pet {tokenizer.mask_token} plays with a ball of string."""

# Encode the input sentence and get the model's output
input = tokenizer.encode(sentence, return_tensors="pt").to(device)
# The model outputs the masked language modeling logits of shape
# [batch_size, sequence_length, vocab_size]
token_logits = model(input)[0]
token_logits


# Activity 2

Try changing the sentence context around the MLM-predicted word.  What types of other things can you get predicted?
